# Практическое задание 1. Обучение полносвязной нейронной сети

Вопросы по работе: **@aloschilov**.

Основная часть: **20 баллов**. Бонусные задания и их баллы указаны в разделе 4.

Это notebook с заданиями: заполните места `YOUR CODE HERE`. До заполнения они намеренно останавливаются с `NotImplementedError`.

In [ ]:
import numpy as np
import torch

from glob import glob
from collections import OrderedDict
from matplotlib import pyplot as plt

from torch import nn
from torch.autograd import Function
from torch.autograd import gradcheck
from torch.optim import Optimizer, Adam
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets
from torchvision import transforms

# Для воспроизводимых численных проверок явно используйте CPU и float64.
# Основное обучение: float32; приоритет MPS -> CUDA -> CPU.
device = torch.device(
    "mps" if torch.backends.mps.is_built() and torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available() else "cpu"
)
print("Устройство обучения:", device)

## 1. Загрузка данных (0 баллов)

Если вам требуется работать с каким-нибудь набором данных (dataset), то, прежде всего, проверьте, нет ли его среди встроенных наборов данных https://pytorch.org/vision/stable/datasets.html.

В текущем домашнем задании мы будем работать с набором данных FashionMNIST. Он присутствует в списке встроенных наборов данных, однако мы воспользуемся реализацией только для удобного и быстрого способа скачать наборы данных. Ниже предлагается реализовать собственный класс для считывания, обработки и упаковки данных.

In [ ]:
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True
)

Воспользуемся функцией загрузки данных из репозитория наборов данных.

In [ ]:
! ls data/FashionMNIST/raw

In [ ]:
#https://github.com/zalandoresearch/fashion-mnist/blob/master/utils/mnist_reader.py

def load_mnist(path, kind='train'):
    import os
    import gzip
    import numpy as np

    """Load MNIST data from `path`"""
    labels_path = os.path.join(path,
                               '%s-labels-idx1-ubyte.gz'
                               % kind)
    images_path = os.path.join(path,
                               '%s-images-idx3-ubyte.gz'
                               % kind)

    with gzip.open(labels_path, 'rb') as lbpath:
        labels = np.frombuffer(lbpath.read(), dtype=np.uint8,
                               offset=8)

    with gzip.open(images_path, 'rb') as imgpath:
        images = np.frombuffer(imgpath.read(), dtype=np.uint8,
                               offset=16).reshape(len(labels), 784)

    return images, labels

Для работы с данными PyTorch предоставляет `Dataset` и `DataLoader` из `torch.utils.data`.

Собственный набор данных реализуют как наследника `Dataset`: метод `__len__` возвращает число примеров, а `__getitem__` возвращает пример по индексу.

`DataLoader` использует этот набор данных, объединяет примеры в батчи и при `num_workers > 0` может загружать их в отдельных процессах. Наследоваться от `DataLoader` для этой работы не требуется.

Реализуем класс для FashionMNIST.

Элементами датасета должны являться пары `(np.ndarray, int)` до применения преобразований, массив имеет размерность `(28, 28)`, тип элемента `np.float32`.

In [ ]:
import os

class FashionMnist(Dataset):
    def __init__(self, path, train=True, image_transform=None,
                 label_transform=None):
        if train:
            images, labels = load_mnist(os.path.join(path,"raw"))
        else:
            images, labels = load_mnist(os.path.join(path,"raw"), kind="t10k")

        ###########################################################
        ############# YOUR CODE HERE ##############################
        ###########################################################
        raise NotImplementedError("Заполните YOUR CODE HERE")

    def __len__(self,):
        ###########################################################
        ############# YOUR CODE HERE ##############################
        ###########################################################
        raise NotImplementedError("Заполните YOUR CODE HERE")
        return length

    def __getitem__(self, idx):
        ###########################################################
        ############# YOUR CODE HERE ##############################
        ###########################################################
        raise NotImplementedError("Заполните YOUR CODE HERE")
        return img, label


In [ ]:
test_dataset = FashionMnist("data/FashionMNIST", train=False)
train_dataset = FashionMnist("data/FashionMNIST")

Визуализируйте случайные элементы набора данных.

In [ ]:
###########################################################
############# YOUR CODE HERE ##############################
###########################################################
raise NotImplementedError("Заполните YOUR CODE HERE")


Наш `FashionMnist` принимает отдельные преобразования изображения и метки через `image_transform` и `label_transform`.

Реализуйте их применение в `FashionMnist`. Ниже дан простой преобразователь: он сохраняет dtype исходного массива и не нормирует значения пикселей. Для изображения подготовьте `np.float32`, для индекса класса используйте целое число. После преобразования метка должна иметь dtype `torch.int64`.

Это не полная копия `torchvision.transforms.ToTensor`: у библиотечного преобразования есть дополнительные правила обработки изображений.

In [ ]:
class ToTensor:
    """Преобразует массив или целую метку в тензор без нормировки."""

    def __call__(self, sample):
        if isinstance(sample, (int, np.integer)):
            return torch.tensor(int(sample), dtype=torch.int64)
        return torch.as_tensor(sample)

In [ ]:
transform = ToTensor()

test_dataset = FashionMnist("data/FashionMNIST",
                            train=False,
                            image_transform=transform,
                            label_transform=transform
                            )
train_dataset = FashionMnist("data/FashionMNIST",
                             image_transform=transform,
                             label_transform=transform
                             )

In [ ]:
print(f"The type of the data is {type(test_dataset[0][0])}")

Элементы набора данных объединяются в батчи. Если стандартное объединение подходит для структуры примера, дополнительную функцию в `DataLoader` передавать не нужно.

In [ ]:
test_dataloader = DataLoader(test_dataset, batch_size=15, num_workers=0, shuffle=True)
batch = next(iter(test_dataloader))

In [ ]:
print(f"The length of the batch is {len(batch)}")
print(f"The shape of the batch[0] is {batch[0].shape}")

Однако, если наша структура данных не позволяет нам использовать объединение по умолчанию, то можно написать собственную функцию, которая будет пакетировать данные.

Реализуйте функцию, преобразующую последовательность элементов массива в пакет (batch).

In [ ]:
def collate(batch):
    ###########################################################
    ############# YOUR CODE HERE ##############################
    ###########################################################
    raise NotImplementedError("Заполните YOUR CODE HERE")
    return imgs, labels

Убедитесь, что все работает корректно.

In [ ]:
test_dataloader = DataLoader(test_dataset, batch_size=15, num_workers=0,
                             shuffle=True, collate_fn=collate)
train_dataloader = DataLoader(train_dataset, batch_size=15, num_workers=0,
                              shuffle=True, collate_fn=collate)
batch = next(iter(test_dataloader))

In [ ]:
print(f"The length of the batch is {len(batch)}")
print(f"The shape of the batch[0] is {batch[0].shape}")

## 2. Реализация модулей нейронной сети (15 баллов)

В этом разделе мы полностью реализуем модули для полносвязной сети.

Для начала нам понадобится реализовать прямой и обратный проход через слои.

Наши слои будут соответствовать следующему интерфейсу (на примере "тождественного" слоя):

Сначала, мы реализуем функцию и её градиент.

In [ ]:
class IdentityFunction(Function):
    """
    We can implement our own custom autograd Functions by subclassing
    torch.autograd.Function and implementing the forward and backward passes
    which operate on Tensors.
    """
    @staticmethod
    def forward(ctx, input):
        """
        In the forward pass we receive a Tensor containing the input and return
        a Tensor containing the output. ctx is a context object that can be used
        to stash information for backward computation. You can cache arbitrary
        objects for use in the backward pass using the ctx.save_for_backward method.
        """
        return input

    @staticmethod
    def backward(ctx, grad_output):
        """
        In the backward pass we receive a Tensor containing the gradient of the loss
        with respect to the output, and we need to compute the gradient of the loss
        with respect to the input.
        """
        return grad_output

Разработанную функцию обернем классом `IdentityLayer`, все слои в `PyTorch` должны быть наследниками базового класса `nn.Module()`


In [ ]:
class IdentityLayer(nn.Module):
    def __init__(self):
        # An identity layer does nothing
        super().__init__()
        self.identity = IdentityFunction.apply

    def forward(self, inp):
        # An identity layer just returns whatever it gets as input.
        return self.identity(inp)


### 2.1 Функция активации ReLU (1 балл)
Для начала реализуем функцию активации, слой нелинейности `ReLU(x) = max(x, 0)`. Параметров у слоя нет. Метод `forward` должен вернуть результат поэлементного применения `ReLU` к входному массиву, метод `backward` - градиент функции потерь по входу слоя. В нуле обычная производная не существует; для обратного прохода выберем значение 0, как в PyTorch.

Для выпуклой функции ReLU субдифференциал в нуле равен $[0,1]$: любое число из этого интервала является допустимым субградиентом. Здесь фиксируем один выбор, а не утверждаем существование обычной производной. Центральная конечная разность в нуле даёт $1/2$, поэтому несовпадение с выбранным значением 0 само по себе не означает ошибку backward. Это пояснение, не дополнительное задание.

 Обратите внимание, что при обратном проходе могут понадобиться величины, посчитанные во время прямого прохода, поэтому их стоит сохранить в `ctx`.

In [ ]:
class ReLUFunction(Function):
    @staticmethod
    def forward(ctx, input):
        ###########################################################
        ############# YOUR CODE HERE ##############################
        ###########################################################
        raise NotImplementedError("Заполните YOUR CODE HERE")
        return

    @staticmethod
    def backward(ctx, grad_output):
        ###########################################################
        ############# YOUR CODE HERE ##############################
        ###########################################################
        raise NotImplementedError("Заполните YOUR CODE HERE")
        return


In [ ]:
class ReLU(nn.Module):
    def __init__(self):
        ###########################################################
        ############# YOUR CODE HERE ##############################
        ###########################################################
        raise NotImplementedError("Заполните YOUR CODE HERE")
        return


    def forward(self, input):
        ###########################################################
        ############# YOUR CODE HERE ##############################
        ###########################################################
        raise NotImplementedError("Заполните YOUR CODE HERE")
        return

После реализации проверьте градиенты с помощью `gradcheck`. Для численной проверки используйте CPU, `torch.float64` и входы с `requires_grad=True`. Проверки выполняйте вдали от точек излома; для ReLU не берите нулевые и слишком близкие к нулю входы. При сравнении со встроенным модулем используйте одинаковые dtype и device.

In [ ]:
###########################################################
############# YOUR CODE HERE ##############################
###########################################################
raise NotImplementedError("Заполните YOUR CODE HERE")

assert gradcheck(relu, x)

In [ ]:
###########################################################
############# YOUR CODE HERE ##############################
###########################################################
raise NotImplementedError("Заполните YOUR CODE HERE")

assert torch.norm(torch_relu(x) - our_relu(x)) < 1e-5

### 2.2 Линейный слой (linear, fully-connected) (3 балла)
Далее реализуем полносвязный слой без нелинейности. У слоя два набора параметров: матрица весов (weights) и вектор смещения (bias).

In [ ]:
class LinearFunction(Function):
    @staticmethod
    def forward(ctx, inp, weight, bias):
        ###########################################################
        ############# YOUR CODE HERE ##############################
        ###########################################################
        raise NotImplementedError("Заполните YOUR CODE HERE")
        return output
    @staticmethod
    def backward(ctx, grad_output):
        ###########################################################
        ############# YOUR CODE HERE ##############################
        ###########################################################
        raise NotImplementedError("Заполните YOUR CODE HERE")
        return grad_input, grad_weight, grad_bias

In [ ]:
class Linear(nn.Module):
    def __init__(self, input_units, output_units):
        super().__init__()
        # initialize weights with small random numbers from normal distribution
        ###########################################################
        ############# YOUR CODE HERE ##############################
        ###########################################################
        raise NotImplementedError("Заполните YOUR CODE HERE")

    def forward(self,inp):
        ###########################################################
        ############# YOUR CODE HERE ##############################
        ###########################################################
        raise NotImplementedError("Заполните YOUR CODE HERE")


Проверим градиенты и сравним работу нашего модуля с `torch.nn.Linear` при одинаковых весах и входах.

Проверка градиента:

In [ ]:
###########################################################
############# YOUR CODE HERE ##############################
###########################################################
raise NotImplementedError("Заполните YOUR CODE HERE")


Сравнение с `PyTorch`.

In [ ]:
###########################################################
############# YOUR CODE HERE ##############################
###########################################################
raise NotImplementedError("Заполните YOUR CODE HERE")

state_dict = OrderedDict([("weight", weight), ("bias", bias)])
torch_linear.load_state_dict(state_dict)
our_linear.load_state_dict(state_dict)

###########################################################
############# YOUR CODE HERE ##############################
###########################################################
raise NotImplementedError("Заполните YOUR CODE HERE")


### 2.3 LogSoftmax (Log + Softmax) (4 балла)

Для многоклассовой классификации `softmax` преобразует логиты $x$ в вероятности классов:

$$
\hat y_i=\frac{\exp(x_i)}{\sum_{j=1}^{K}\exp(x_j)},\qquad i=1,\ldots,K.
$$

Здесь $K$ обозначает число классов. Для одного объекта с one-hot меткой $y$ кросс-энтропия равна отрицательному логарифму правдоподобия:

$$
\ell(y,\hat y)=-\sum_{i=1}^{K}y_i\log\hat y_i.
$$

Если $c$ обозначает индекс истинного класса, ту же потерю можно записать короче:

$$
\ell(c,\hat y)=-\log\hat y_c.
$$

Реализуйте слой `LogSoftmax` без параметров. Метод `forward` вычисляет логарифмы вероятностей, а `backward` вычисляет градиент потерь по входу из входящего `grad_output`.

Явный якобиан для батча был бы трёхмерным тензором, но строить его не обязательно. Сохраняется допущение исходного задания: достаточно поддержать `grad_output`, у которого в каждой строке только один ненулевой элемент (не обязательно единица).

Для полного балла нужна реализация с **Log-Sum-Exp trick**.

In [ ]:
class LogSoftmaxFunction(Function):
    @staticmethod
    def forward(ctx, inp):
        ###########################################################
        ############# YOUR CODE HERE ##############################
        ###########################################################
        raise NotImplementedError("Заполните YOUR CODE HERE")

    @staticmethod
    def backward(ctx, grad_output):
        ###########################################################
        ############# YOUR CODE HERE ##############################
        ###########################################################
        raise NotImplementedError("Заполните YOUR CODE HERE")


In [ ]:
class LogSoftmax(nn.Module):
    def __init__(self):
        super().__init__()
        ###########################################################
        ############# YOUR CODE HERE ##############################
        ###########################################################
        raise NotImplementedError("Заполните YOUR CODE HERE")

    def forward(self, input):
        ###########################################################
        ############# YOUR CODE HERE ##############################
        ###########################################################
        raise NotImplementedError("Заполните YOUR CODE HERE")


Проверка градиентов.

In [ ]:
###########################################################
############# YOUR CODE HERE ##############################
###########################################################
raise NotImplementedError("Заполните YOUR CODE HERE")


### 2.4 Dropout (2 балла)
Реализуйте слой Dropout. Здесь `p` означает вероятность зануления. Поведение в режимах `train()` и `eval()` должно соответствовать `torch.nn.Dropout`.

При численной проверке случайная маска должна оставаться одной и той же для всех вычислений сравниваемой функции. Иначе конечная разность измеряет также изменение маски и не проверяет производную.

In [ ]:
class DropoutFunction(Function):
    @staticmethod
    def forward(ctx, inp, p):
        ###########################################################
        ############# YOUR CODE HERE ##############################
        ###########################################################
        raise NotImplementedError("Заполните YOUR CODE HERE")

    @staticmethod
    def backward(ctx, grad_output):
        ###########################################################
        ############# YOUR CODE HERE ##############################
        ###########################################################
        raise NotImplementedError("Заполните YOUR CODE HERE")


In [ ]:
class Dropout(nn.Module):
    def __init__(self, p):
        super().__init__()
        ###########################################################
        ############# YOUR CODE HERE ##############################
        ###########################################################
        raise NotImplementedError("Заполните YOUR CODE HERE")

    def forward(self, input):
        ###########################################################
        ############# YOUR CODE HERE ##############################
        ###########################################################
        raise NotImplementedError("Заполните YOUR CODE HERE")


### 2.5 CrossEntropy (5 баллов)

При решении задачи многоклассовой классификации мы будем использовать в качестве функции потерь **кроссэнтропию, совместимую с `LogSoftmax` активацией**.

На вход поступают **логарифмы вероятностей** из вашего `LogSoftmax`, а не логиты. По этому интерфейсу функция соответствует `torch.nn.NLLLoss`, а не `torch.nn.CrossEntropyLoss`, которая сама включает LogSoftmax. Возвращайте среднюю потерю по батчу, как ожидает приведённый цикл обучения. Метки представляются индексами классов (`torch.int64`).

Реализуйте эту функцию потерь. В разделе 2.3 приведены полезные формулы.

In [ ]:
class CrossEntropyFunction(Function):
    @staticmethod
    def forward(ctx, activations, target):
        ###########################################################
        ############# YOUR CODE HERE ##############################
        ###########################################################
        raise NotImplementedError("Заполните YOUR CODE HERE")

    @staticmethod
    def backward(ctx, grad_output):
        ###########################################################
        ############# YOUR CODE HERE ##############################
        ###########################################################
        raise NotImplementedError("Заполните YOUR CODE HERE")

class CrossEntropy(nn.Module):
    def __init__(self, ):
        super().__init__()
        ###########################################################
        ############# YOUR CODE HERE ##############################
        ###########################################################
        raise NotImplementedError("Заполните YOUR CODE HERE")

    def forward(self, activations, target):
        ###########################################################
        ############# YOUR CODE HERE ##############################
        ###########################################################
        raise NotImplementedError("Заполните YOUR CODE HERE")


Проверка градиентов.

In [ ]:
###########################################################
############# YOUR CODE HERE ##############################
###########################################################
raise NotImplementedError("Заполните YOUR CODE HERE")


## 3. Сборка и обучение нейронной сети (5 баллов)

**В этой работе стандартный тестовый набор FashionMNIST (`train=False`, `test_dataloader`) используется как валидационная выборка для оценки loss и accuracy, наблюдения за обучением и сравнения моделей. Используйте его во всех экспериментах; отдельное разбиение создавать не требуется. Независимая тестовая оценка после выбора модели в эту работу не входит.**

Реализуйте из ваших блоков персептрон и обучите его, записав итоговую функцию потерь и accuracy на валидационной выборке. **(1 балл)**

Подсказка: вытягиваем картинку в вектор с помощью [nn.Flatten](https://pytorch.org/docs/stable/generated/torch.nn.Flatten.html)

In [ ]:
class Network(nn.Module):
    def __init__(self, input_size=28*28, hidden_layers_size=32, num_layers=5,
                 num_classes=10):
        super().__init__()
        ###########################################################
        ############# YOUR CODE HERE ##############################
        ###########################################################
        raise NotImplementedError("Заполните YOUR CODE HERE")

    def forward(self, inp):
        ###########################################################
        ############# YOUR CODE HERE ##############################
        ###########################################################
        raise NotImplementedError("Заполните YOUR CODE HERE")
    def predict(self, inp):
        ###########################################################
        ############# YOUR CODE HERE ##############################
        ###########################################################
        raise NotImplementedError("Заполните YOUR CODE HERE")


Ниже приведены функции, реализующие обучение нейронной сети. В данном задании их предлагается просто переиспользовать.

In [ ]:
class EmptyContext:
    def __enter__(self):
        pass

    def __exit__(self, *args):
        pass

In [ ]:
# accuracy metric for our classififcation
def accuracy(model_labels, labels):
  return torch.mean((model_labels == labels).float())

In [ ]:
def perform_epoch(model, loader, criterion, optimizer=None, device=None):
    is_train = optimizer is not None
    model = model.to(device)
    model.train(is_train)
    total_loss = 0.0
    total_correct = 0
    total_n = 0
    with EmptyContext() if is_train else torch.no_grad():
        for batch_data, batch_labels in loader:
            batch_data = batch_data.to(device)
            batch_labels = batch_labels.to(device)
            if is_train:
                optimizer.zero_grad()
            model_labels = model(batch_data)
            new_loss = criterion(model_labels, batch_labels)
            model_prediction = model_labels.detach().argmax(dim=1)
            if is_train:
                new_loss.backward()
                optimizer.step()
            batch_n = batch_labels.shape[0]
            total_loss += new_loss.detach().item() * batch_n
            total_correct += (model_prediction == batch_labels).sum().item()
            total_n += batch_n
    if total_n == 0:
        raise ValueError("Набор данных не должен быть пустым")
    return total_loss / total_n, total_correct / total_n

Теперь обучим нашу нейронную сеть. В данном разделе будем использовать оптимизатор `Adam` с параметрами по умолчанию.

In [ ]:
# YOUR CODE HERE: создайте model и criterion, затем optimizer Adam.
# До создания optimizer перенесите model на device в torch.float32.
raise NotImplementedError("Создайте модель, функцию потерь и оптимизатор")

In [ ]:
for epoch in range(10):
    loss, acc = perform_epoch(model, train_dataloader, criterion,
                                optimizer=optimizer, device=device)
    print(f"Epoch - {epoch} : loss {loss}, accuracy {acc}")
    print(f"Current learning rate: {optimizer.param_groups[0]['lr']}")

In [ ]:
print(f"Epoch - {epoch} : loss {loss}, accuracy {acc}")
print(f"Current learning rate: {optimizer.param_groups[0]['lr']}")

Дальше **(4 балла)**:
- Проведите эксперименты с числом слоев.
- Постройте графики зависимости качества модели на обучающей и валидационной выборках от числа слоев. Для получения статистически значимых результатов повторите эксперименты несколько раз.
- Сделайте выводы.

Training Loop для выполнения этой части задания можно и нужно улучшать, в том числе, добавляя более продвинутое логгирование эксперимента.

## 4. Бонусная часть.

### 4.1 Реализация метода оптимизации (3 + 3 балла).
Реализуйте сами метод оптимизации  для рассмотренной выше архитектуры. Вы можете выбрать произвольный метод от градиентного спуска до современных вариантов. Продемонстрируйте правильную работу метода оптимизации, сравните его работу с Adam.

**Дополнительные баллы** вы получите, если метод будет уникален среди сдавших задание.

In [ ]:
class SotaOptimizer(Optimizer):
    def __init__(self, params, lr=1e-3):
        defaults = dict(lr=lr)
        super(SotaOptimizer, self).__init__(params, defaults)

    def __setstate__(self, state):
        super(SotaOptimizer, self).__setstate__(state)

    @torch.no_grad()
    def step(self,):

        for group in self.param_groups:
            lr = group['lr']
            for p in group['params']:
                if p.grad is not None:
                    p.add_(p.grad, alpha=-lr)

### 4.2 Реализация современной функции активации (2 + 2 балла).
Реализуйте одну из активаций, предложенных на лекции или в статье. Например, `Hardswish`. Сравните сеть с вашей активацией и с `ReLU`.

**Дополнительные баллы** вы получите, если функция будет уникальна среди сдавших задание.